In [6]:
# ============================================================
# BANK ACCOUNT SIMULATOR
# Demonstrates: Encapsulation, Inheritance,
#               Polymorphism, Abstraction
# ============================================================

from abc import ABC, abstractmethod
from datetime import datetime

# ============================================================
# ABSTRACT BASE CLASS — Abstraction
# ============================================================
class Account(ABC):

    def __init__(self, account_number, owner_name, balance=0):
        self.__account_number = account_number  # Encapsulation
        self.__owner_name = owner_name          # Encapsulation
        self.__balance = balance                # Encapsulation
        self.__transactions = []               # Encapsulation

    # Getters — controlled access to private data
    def get_account_number(self):
        return self.__account_number

    def get_owner_name(self):
        return self.__owner_name

    def get_balance(self):
        return self.__balance

    def get_transactions(self):
        return self.__transactions

    # Deposit method
    def deposit(self, amount):
        if amount <= 0:
            print("Deposit amount must be positive!")
            return False
        self.__balance += amount
        self.__transactions.append({
            'type': 'Deposit',
            'amount': amount,
            'date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'balance': self.__balance
        })
        print(f"Deposited ₹{amount:,.2f} | New Balance: ₹{self.__balance:,.2f}")
        return True

    # Withdrawal method
    def withdraw(self, amount):
        if amount <= 0:
            print("Withdrawal amount must be positive!")
            return False
        if amount > self.__balance:
            print(f"Insufficient funds! Available: ₹{self.__balance:,.2f}")
            return False
        self.__balance -= amount
        self.__transactions.append({
            'type': 'Withdrawal',
            'amount': amount,
            'date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'balance': self.__balance
        })
        print(f"Withdrawn ₹{amount:,.2f} | New Balance: ₹{self.__balance:,.2f}")
        return True

    # Transfer method
    def transfer(self, amount, target_account):
        print(f"\nTransferring ₹{amount:,.2f} to {target_account.get_owner_name()}...")
        if self.withdraw(amount):
            target_account.deposit(amount)
            print(f"Transfer successful!")
        else:
            print(f"Transfer failed!")

    # Transaction history
    def print_statement(self):
        print(f"\n{'='*55}")
        print(f"  ACCOUNT STATEMENT — {self.get_owner_name()}")
        print(f"  Account No: {self.get_account_number()}")
        print(f"  Account Type: {self.get_account_type()}")
        print(f"{'='*55}")
        if not self.__transactions:
            print("  No transactions yet.")
        for t in self.__transactions:
            print(f"  {t['date']} | {t['type']:<12} | "
                  f"₹{t['amount']:>10,.2f} | Balance: ₹{t['balance']:>10,.2f}")
        print(f"{'='*55}")
        print(f"  Current Balance: ₹{self.__balance:,.2f}")
        print(f"{'='*55}\n")

    # Abstract method — must be implemented by subclasses
    @abstractmethod
    def get_account_type(self):
        pass

    @abstractmethod
    def apply_interest(self):
        pass

print("Base Account class defined!")

Base Account class defined!


In [7]:
# ============================================================
# SAVINGS ACCOUNT — Inheritance + Polymorphism
# ============================================================
class SavingsAccount(Account):

    INTEREST_RATE = 0.04  # 4% annual interest

    def __init__(self, account_number, owner_name, balance=0):
        super().__init__(account_number, owner_name, balance)
        print(f"Savings Account created for {owner_name}")

    # Polymorphism — own version of abstract methods
    def get_account_type(self):
        return "Savings Account"

    def apply_interest(self):
        interest = self.get_balance() * self.INTEREST_RATE
        self.deposit(interest)
        print(f"Interest of 4% applied: ₹{interest:,.2f} added!")

# ============================================================
# CURRENT ACCOUNT — Inheritance + Polymorphism
# ============================================================
class CurrentAccount(Account):

    INTEREST_RATE = 0.02  # 2% annual interest
    OVERDRAFT_LIMIT = 10000  # Can go ₹10,000 below zero

    def __init__(self, account_number, owner_name, balance=0):
        super().__init__(account_number, owner_name, balance)
        print(f"Current Account created for {owner_name}")

    # Polymorphism — overrides withdraw to allow overdraft
    def withdraw(self, amount):
        if amount <= 0:
            print("Withdrawal amount must be positive!")
            return False
        if amount > self.get_balance() + self.OVERDRAFT_LIMIT:
            print(f"Exceeds overdraft limit! Max withdrawal: "
                  f"₹{self.get_balance() + self.OVERDRAFT_LIMIT:,.2f}")
            return False
        # Use parent withdraw if enough balance
        return super().withdraw(amount)

    def get_account_type(self):
        return "Current Account"

    def apply_interest(self):
        interest = self.get_balance() * self.INTEREST_RATE
        self.deposit(interest)
        print(f"Interest of 2% applied: ₹{interest:,.2f} added!")

print("SavingsAccount and CurrentAccount classes defined!")

SavingsAccount and CurrentAccount classes defined!


In [8]:
# ============================================================
# BANK CLASS — Manages all accounts
# ============================================================
class Bank:

    def __init__(self, bank_name):
        self.__bank_name = bank_name
        self.__accounts = {}  # stores all accounts
        print(f"{self.__bank_name} initialized!")

    def create_savings_account(self, owner_name, initial_balance=0):
        account_number = f"SAV{len(self.__accounts)+1:04d}"
        account = SavingsAccount(account_number, owner_name, initial_balance)
        self.__accounts[account_number] = account
        print(f"Account Number: {account_number}")
        return account

    def create_current_account(self, owner_name, initial_balance=0):
        account_number = f"CUR{len(self.__accounts)+1:04d}"
        account = CurrentAccount(account_number, owner_name, initial_balance)
        self.__accounts[account_number] = account
        print(f"Account Number: {account_number}")
        return account

    def get_account(self, account_number):
        return self.__accounts.get(account_number, None)

    def print_all_accounts(self):
        print(f"\n{'='*55}")
        print(f"  {self.__bank_name} — All Accounts")
        print(f"{'='*55}")
        for acc_num, acc in self.__accounts.items():
            print(f"  {acc_num} | {acc.get_owner_name():<20} | "
                  f"{acc.get_account_type():<20} | "
                  f"₹{acc.get_balance():>10,.2f}")
        print(f"{'='*55}\n")

print("Bank class defined!")

Bank class defined!


In [10]:
# ============================================================
# SIMULATION — Testing all OOP concepts
# ============================================================

print("=" * 55)
print("   WELCOME TO NATIONAL BANK OF INDIA SIMULATOR")
print("=" * 55)

# Create bank
bank = Bank("National Bank of India")

print("\n--- Creating Accounts ---")
# Create accounts — Inheritance in action
acc1 = bank.create_savings_account("Mridusmitha", 50000)
acc2 = bank.create_savings_account("Rahul Sharma", 30000)
acc3 = bank.create_current_account("Tech Startup Ltd", 100000)

print("\n--- Deposits ---")
# Deposits — Encapsulation in action
acc1.deposit(10000)
acc2.deposit(5000)
acc3.deposit(25000)

print("\n--- Withdrawals ---")
# Withdrawals
acc1.withdraw(8000)
acc2.withdraw(50000)  # Should fail — insufficient funds
acc3.withdraw(120000) # Should work — overdraft allowed

print("\n--- Transfer ---")
# Transfer between accounts
acc1.transfer(5000, acc2)

print("\n--- Applying Interest ---")
# Polymorphism in action — each account type ap

   WELCOME TO NATIONAL BANK OF INDIA SIMULATOR
National Bank of India initialized!

--- Creating Accounts ---
Savings Account created for Mridusmitha
Account Number: SAV0001
Savings Account created for Rahul Sharma
Account Number: SAV0002
Current Account created for Tech Startup Ltd
Account Number: CUR0003

--- Deposits ---
Deposited ₹10,000.00 | New Balance: ₹60,000.00
Deposited ₹5,000.00 | New Balance: ₹35,000.00
Deposited ₹25,000.00 | New Balance: ₹125,000.00

--- Withdrawals ---
Withdrawn ₹8,000.00 | New Balance: ₹52,000.00
Insufficient funds! Available: ₹35,000.00
Withdrawn ₹120,000.00 | New Balance: ₹5,000.00

--- Transfer ---

Transferring ₹5,000.00 to Rahul Sharma...
Withdrawn ₹5,000.00 | New Balance: ₹47,000.00
Deposited ₹5,000.00 | New Balance: ₹40,000.00
Transfer successful!

--- Applying Interest ---
